In [ ]:
!pip install mcp pywin32 FastMCP

In [ ]:
# Cell 2: Create the server code (weather.py)
# We'll write the server file to disk so the subprocess can run it.

server_code = """
from fastmcp import FastMCP
import requests

app = FastMCP("WeatherServer")

@app.tool()
def get_weather(city: str) -> str:
    \"""Get current weather for a city.\"""
    response = requests.get(f"https://wttr.in/{city}?format=3")
    return response.text

if __name__ == "__main__":
   app.run(transport="sse", host="127.0.0.1", port=8000)

"""

with open("weather.py", "w") as f:
    f.write(server_code)

print("weather.py written")

In [ ]:
!python weather.py

In [ ]:
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession

async def main():
    async with sse_client("http://localhost:8000/sse") as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools:", tools)
            # Call tool
            result = await session.call_tool("get_weather", {"city": "Pune"})
            print("Result:", result)

await main()